# 1. Credit Card Fraud Detection

This notebook builds an educational machine learning system for detecting potentially fraudulent credit card transactions. It uses the provided `creditcard.csv` dataset, SMOTE, Random Forest, XGBoost, validation-based model selection, and an untouched final test set.


## 2. Project Objective

The goal is to compare Random Forest and XGBoost on a highly imbalanced fraud dataset. Exact duplicates are removed before splitting, `Time` and `Amount` are scaled, and SMOTE is applied only to development training data to prevent leakage.

`Class = 0` means a normal transaction and `Class = 1` means a fraudulent transaction. This is an educational project, not a production banking decision system.


## 3. Import Libraries


In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 35)

RANDOM_STATE = 42
TARGET_COL = "Class"
SCALE_COLS = ["Time", "Amount"]
FEATURE_NAMES = ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount"]
METRIC_NAMES = ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_PATH = PROJECT_DIR / "data" / "creditcard.csv"
MODEL_DIR = PROJECT_DIR / "models"


## 4. Load Dataset


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "creditcard.csv not found. Place it inside the data folder."
    )

df = pd.read_csv(DATA_PATH)
print(f"Dataset path: {DATA_PATH}")
print(f"Original shape: {df.shape}")


## 5. Dataset Overview

`V1` to `V28` are anonymized PCA-derived numerical features. Their original business meanings are not available. `Time` is elapsed transaction time, `Amount` is the transaction amount, and `Class` is the target.


In [ ]:
required_cols = FEATURE_NAMES + [TARGET_COL]
missing_cols = [col for col in required_cols if col not in df.columns]
extra_cols = [col for col in df.columns if col not in required_cols]

if missing_cols:
    raise ValueError(f"Dataset is missing required columns: {missing_cols}")
if extra_cols:
    raise ValueError(f"Dataset contains unexpected columns: {extra_cols}")

non_numeric_cols = df[FEATURE_NAMES].select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric_cols:
    raise ValueError(f"Input features must be numerical: {non_numeric_cols}")

target_values = set(df[TARGET_COL].dropna().unique().tolist())
if target_values != {0, 1}:
    raise ValueError(f"Class must contain only 0 and 1. Found: {sorted(target_values)}")

df[TARGET_COL] = df[TARGET_COL].astype(int)
display(df.head())
display(df.describe().T)


## 6. Missing Values


In [ ]:
missing_by_column = df.isnull().sum()
missing_total = int(missing_by_column.sum())
print(f"Total missing values: {missing_total:,}")

if missing_total > 0:
    display(missing_by_column[missing_by_column > 0].to_frame("Missing Values"))
    raise ValueError("Missing values must be handled before training.")


## 7. Duplicate Analysis

Exact duplicates are checked before splitting. Leaving identical rows in the dataset could place the same record in both training and evaluation data and make results less reliable.


In [ ]:
original_rows = len(df)
duplicate_rows = int(df.duplicated().sum())
print(f"Original rows: {original_rows:,}")
print(f"Exact duplicate rows: {duplicate_rows:,}")


## 8. Remove Exact Duplicates


In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
print(f"Cleaned shape: {df.shape}")
print(f"Rows removed: {original_rows - len(df):,}")


## 9. Class Distribution


In [ ]:
class_counts = df[TARGET_COL].value_counts().sort_index()
class_percentages = (class_counts / len(df) * 100).rename("Percentage")

class_summary = pd.DataFrame({
    "Class Label": ["Normal", "Fraud"],
    "Count": [int(class_counts.get(0, 0)), int(class_counts.get(1, 0))],
    "Percentage": [
        float(class_percentages.get(0, 0.0)),
        float(class_percentages.get(1, 0.0)),
    ],
}, index=[0, 1])

display(class_summary.style.format({"Count": "{:,}", "Percentage": "{:.4f}%"}))


## 10. Exploratory Data Analysis

The plots below focus on class imbalance and transaction amount. Amount plots are limited to the 99th percentile for readability; no business meaning is assigned to the anonymized PCA features.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(data=df, x=TARGET_COL, ax=axes[0, 0], color="#4C72B0")
axes[0, 0].set_yscale("log")
axes[0, 0].set_title("Class Distribution (Log Scale)")
axes[0, 0].set_xticklabels(["Normal", "Fraud"])

axes[0, 1].bar(
    ["Normal", "Fraud"],
    [class_percentages.get(0, 0.0), class_percentages.get(1, 0.0)],
    color=["#55A868", "#C44E52"],
)
axes[0, 1].set_title("Class Distribution Percentage")
axes[0, 1].set_ylabel("Percentage")

amount_limit = df["Amount"].quantile(0.99)
sns.histplot(
    data=df[df["Amount"] <= amount_limit],
    x="Amount",
    hue=TARGET_COL,
    bins=60,
    element="step",
    stat="density",
    common_norm=False,
    ax=axes[1, 0],
)
axes[1, 0].set_title("Amount Distribution up to 99th Percentile")

sns.boxplot(data=df, x=TARGET_COL, y="Amount", showfliers=False, ax=axes[1, 1])
axes[1, 1].set_title("Amount Comparison by Class")
axes[1, 1].set_xticklabels(["Normal", "Fraud"])

plt.tight_layout()
plt.show()


## 11. Feature and Target Separation


In [ ]:
X = df[FEATURE_NAMES].copy()
y = df[TARGET_COL].copy()

assert list(X.columns) == FEATURE_NAMES
assert TARGET_COL not in X.columns
assert set(y.unique()) == {0, 1}

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Number of input features: {X.shape[1]}")


## 12. Train / Validation / Test Split

The final test set is separated first and kept untouched during model comparison. Stratification preserves the rare fraud class in all three splits.


In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.20,
    stratify=y_train_val,
    random_state=RANDOM_STATE,
)

X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

split_summary = pd.DataFrame({
    "Split": ["Training", "Validation", "Final Test"],
    "Rows": [len(X_train), len(X_val), len(X_test)],
    "Fraud Records": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
})
display(split_summary)


## 13. Scale Time and Amount

`V1` to `V28` are already PCA-transformed. Only `Time` and `Amount` are scaled because SMOTE uses distances. The scaler is fitted on training data only and only transforms validation data.


In [ ]:
initial_scaler = StandardScaler()
initial_scaler.fit(X_train[SCALE_COLS])

X_train_processed = X_train.copy()
X_val_processed = X_val.copy()

X_train_processed[SCALE_COLS] = initial_scaler.transform(X_train[SCALE_COLS])
X_val_processed[SCALE_COLS] = initial_scaler.transform(X_val[SCALE_COLS])

X_train_processed = X_train_processed[FEATURE_NAMES]
X_val_processed = X_val_processed[FEATURE_NAMES]

print("Training Time mean after scaling:", round(X_train_processed["Time"].mean(), 10))
print("Training Amount mean after scaling:", round(X_train_processed["Amount"].mean(), 10))


## 14. Class Distribution Before SMOTE


In [ ]:
before_smote = y_train.value_counts().sort_index()
print("Before SMOTE")
print(f"Normal: {int(before_smote.get(0, 0)):,}")
print(f"Fraud:  {int(before_smote.get(1, 0)):,}")


## 15. Apply SMOTE

SMOTE creates synthetic minority-class examples so the models see more fraud patterns during training. It is applied only after splitting and only to processed training data.


In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train_processed, y_train)

if not isinstance(X_train_smote, pd.DataFrame):
    X_train_smote = pd.DataFrame(X_train_smote, columns=FEATURE_NAMES)
else:
    X_train_smote = X_train_smote[FEATURE_NAMES]

if not isinstance(y_train_smote, pd.Series):
    y_train_smote = pd.Series(y_train_smote, name=TARGET_COL)


## 16. Class Distribution After SMOTE


In [ ]:
after_smote = y_train_smote.value_counts().sort_index()
print("After SMOTE")
print(f"Normal: {int(after_smote.get(0, 0)):,}")
print(f"Fraud:  {int(after_smote.get(1, 0)):,}")


## 17. Random Forest

Random Forest combines multiple decision trees. Tree depth is limited to reduce unnecessary model size and overfitting risk.


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_model.fit(X_train_smote, y_train_smote)
print("Random Forest training complete.")


## 18. Random Forest Validation Results

Accuracy alone is misleading because fraud represents far less than 1% of the data. Recall shows how many actual frauds were detected, while F1 balances precision and recall. A false negative is an actual fraud predicted as normal.


In [ ]:
def evaluate_model(model, X_data, y_data, model_name):
    predictions = model.predict(X_data)
    class_labels = list(model.classes_)
    fraud_class_index = class_labels.index(1)
    fraud_probabilities = model.predict_proba(X_data)[:, fraud_class_index]

    metrics = {
        "Model": model_name,
        "Accuracy": float(accuracy_score(y_data, predictions)),
        "Precision": float(precision_score(y_data, predictions, pos_label=1, zero_division=0)),
        "Recall": float(recall_score(y_data, predictions, pos_label=1, zero_division=0)),
        "F1 Score": float(f1_score(y_data, predictions, pos_label=1, zero_division=0)),
        "ROC-AUC": float(roc_auc_score(y_data, fraud_probabilities)),
    }
    matrix = confusion_matrix(y_data, predictions, labels=[0, 1])
    report = classification_report(
        y_data,
        predictions,
        labels=[0, 1],
        target_names=["Normal", "Fraud"],
        zero_division=0,
    )
    return metrics, predictions, fraud_probabilities, matrix, report


rf_metrics, rf_pred, rf_prob, rf_matrix, rf_report = evaluate_model(
    rf_model,
    X_val_processed,
    y_val,
    "Random Forest",
)

display(pd.DataFrame([rf_metrics]))
print(rf_report)


## 19. XGBoost

XGBoost builds trees sequentially, with each new tree learning from earlier errors. The configuration is kept CPU-friendly and understandable.


In [ ]:
xgb_model = XGBClassifier(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    eval_metric="logloss",
    n_jobs=-1,
)
xgb_model.fit(X_train_smote, y_train_smote)
print("XGBoost training complete.")


## 20. XGBoost Validation Results


In [ ]:
xgb_metrics, xgb_pred, xgb_prob, xgb_matrix, xgb_report = evaluate_model(
    xgb_model,
    X_val_processed,
    y_val,
    "XGBoost",
)

display(pd.DataFrame([xgb_metrics]))
print(xgb_report)


## 21. Confusion Matrices

`TN` = normal correctly predicted, `FP` = normal incorrectly predicted as fraud, `FN` = fraud incorrectly predicted as normal, and `TP` = fraud correctly predicted.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for matrix, title, axis in [
    (rf_matrix, "Random Forest - Validation", axes[0]),
    (xgb_matrix, "XGBoost - Validation", axes[1]),
]:
    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=["Normal", "Fraud"],
        yticklabels=["Normal", "Fraud"],
        ax=axis,
    )
    axis.set_title(title)
    axis.set_xlabel("Predicted Label")
    axis.set_ylabel("Actual Label")

plt.tight_layout()
plt.show()


## 22. Model Comparison


In [ ]:
validation_results = [rf_metrics, xgb_metrics]
comparison = pd.DataFrame(validation_results)[["Model"] + METRIC_NAMES]
display(comparison.style.format({metric: "{:.6f}" for metric in METRIC_NAMES}))


## 23. Select Best Algorithm

F1 Score is the main selection metric. When F1 scores are extremely close, Recall is considered next, followed by ROC-AUC. The final test set is not used for this decision.


In [ ]:
rf_result = validation_results[0]
xgb_result = validation_results[1]
close_tolerance = 0.001

if not np.isclose(rf_result["F1 Score"], xgb_result["F1 Score"], atol=close_tolerance):
    selection_metric = "F1 Score"
elif not np.isclose(rf_result["Recall"], xgb_result["Recall"], atol=close_tolerance):
    selection_metric = "Recall"
else:
    selection_metric = "ROC-AUC"

selected_result = max(validation_results, key=lambda result: result[selection_metric])
selected_model_name = selected_result["Model"]

print(f"Selection decided using: {selection_metric}")
print(f"Selected algorithm: {selected_model_name}")


## 24. Final Model Training

After algorithm selection, the original non-SMOTE training and validation samples are combined. A new scaler is fitted on this development data, a new SMOTE instance is applied only to it, and a fresh selected model is trained.


In [ ]:
X_development = pd.concat([X_train, X_val], ignore_index=True)
y_development = pd.concat([y_train, y_val], ignore_index=True)

final_scaler = StandardScaler()
final_scaler.fit(X_development[SCALE_COLS])

X_development_processed = X_development.copy()
X_development_processed[SCALE_COLS] = final_scaler.transform(
    X_development[SCALE_COLS]
)
X_development_processed = X_development_processed[FEATURE_NAMES]

final_smote = SMOTE(random_state=RANDOM_STATE)
X_development_smote, y_development_smote = final_smote.fit_resample(
    X_development_processed,
    y_development,
)

if selected_model_name == "Random Forest":
    final_model = RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
else:
    final_model = XGBClassifier(
        n_estimators=150,
        max_depth=5,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        n_jobs=-1,
    )

final_model.fit(X_development_smote, y_development_smote)
print(f"Final {selected_model_name} training complete.")


## 25. Final Test Evaluation

The untouched test set is transformed with the final scaler and evaluated once. SMOTE is never applied to this test set.


In [ ]:
X_test_processed = X_test.copy()
X_test_processed[SCALE_COLS] = final_scaler.transform(X_test[SCALE_COLS])
X_test_processed = X_test_processed[FEATURE_NAMES]

test_metrics, test_pred, test_prob, test_matrix, test_report = evaluate_model(
    final_model,
    X_test_processed,
    y_test,
    selected_model_name,
)

display(pd.DataFrame([test_metrics]))
print(test_report)

plt.figure(figsize=(6, 4))
sns.heatmap(
    test_matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Normal", "Fraud"],
    yticklabels=["Normal", "Fraud"],
)
plt.title(f"{selected_model_name} - Final Test Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.tight_layout()
plt.show()


## 26. Save Model Artifacts


In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "fraud_model.pkl"
scaler_path = MODEL_DIR / "scaler.pkl"
feature_names_path = MODEL_DIR / "feature_names.pkl"
model_info_path = MODEL_DIR / "model_info.json"

joblib.dump(final_model, model_path, compress=3)
joblib.dump(final_scaler, scaler_path, compress=3)
joblib.dump(FEATURE_NAMES, feature_names_path, compress=3)

validation_metrics = {
    result["Model"]: {metric: result[metric] for metric in METRIC_NAMES}
    for result in validation_results
}
final_test_metrics = {metric: test_metrics[metric] for metric in METRIC_NAMES}

model_info = {
    "model_name": selected_model_name,
    "target_column": TARGET_COL,
    "normal_class": 0,
    "fraud_class": 1,
    "scaled_columns": SCALE_COLS,
    "feature_count": len(FEATURE_NAMES),
    "selection_rule": "F1 Score, then Recall, then ROC-AUC",
    "validation_metrics": validation_metrics,
    "test_metrics": final_test_metrics,
}

with model_info_path.open("w", encoding="utf-8") as file:
    json.dump(model_info, file, indent=4)

print("Saved files:")
print(model_path)
print(scaler_path)
print(feature_names_path)
print(model_info_path)


## 27. Conclusion

This workflow removes exact duplicates, preserves an untouched test set, scales only `Time` and `Amount`, applies SMOTE only to development training data, compares Random Forest and XGBoost using validation metrics, and saves the final retrained model with its preprocessing artifacts.

Recall and F1 Score are more informative than Accuracy for this highly imbalanced problem. A false negative means an actual fraudulent transaction was missed. The saved model is suitable for the educational Streamlit application in this project, but it should not be treated as a production banking system.
